### Загрузка исходного датасета

In [ ]:
import pandas as pd

DATA_PATH = "data/shopease_raw_orders.csv"
df = pd.read_csv(DATA_PATH)
df.head()

### Оценка пропусков перед очисткой

In [ ]:
missing_before = pd.DataFrame({
    "Количество пропусков": df.isna().sum(),
    "Доля пропусков, %": df.isna().mean().mul(100)
}).sort_values("Доля пропусков, %", ascending=False)

display(missing_before)

### Создание копии для очистки

In [ ]:
rows_before = len(df)
df_clean = df.dropna(subset=["OrderID", "CustomerID"]).copy()
print(f"Удалено объектов без OrderID или CustomerID: {rows_before - len(df_clean)}")

### Заполнение пропусков в числовых признаках медианой

In [ ]:
# Списки признаков заданы явно: Rating сохраняем без заполнения,
# так как его ожидаемость зависит от статуса заказа.
NUMERIC_COLS = ["CustomerAge", "Quantity", "UnitPrice", "TotalAmount"]

for col in NUMERIC_COLS:
    if col in df_clean.columns:
        values = pd.to_numeric(df_clean[col], errors="coerce")
        df_clean[col] = values.fillna(values.median())

# Для Discount пропуски заполняются нулём.
if "Discount" in df_clean.columns:
    df_clean["DiscountWasMissing"] = df_clean["Discount"].isna()
    discount_raw = df_clean["Discount"].astype("string").str.strip()
    discount_values = pd.to_numeric(discount_raw.str.rstrip("%"), errors="coerce")
    percent_mask = discount_raw.str.endswith("%", na=False)
    discount_values.loc[percent_mask] /= 100
    df_clean["Discount"] = discount_values.fillna(0)


### Добавление флагов ожидаемости оценки и даты доставки

In [ ]:
status_normalized = df_clean["OrderStatus"].astype("string").str.strip().str.lower()
delivered_mask = status_normalized.eq("delivered").fillna(False)

df_clean["RatingExpected"] = delivered_mask
df_clean["DeliveryDateExpected"] = delivered_mask

display(df_clean[["OrderStatus", "Rating", "RatingExpected", "DeliveryDate", "DeliveryDateExpected"]].head(10))

### Обозначение пропусков в категориальных признаках

In [ ]:
CATEGORICAL_COLS = [
    "Gender", "City", "Category", "Product", "PaymentMethod", "OrderStatus"
]

for col in CATEGORICAL_COLS:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna("Не указано")

### Проверка незаполненных пропусков

In [ ]:
missing_after = pd.DataFrame({
    "Количество пропусков": df_clean.isna().sum(),
    "Доля пропусков, %": df_clean.isna().mean().mul(100)
}).sort_values("Доля пропусков, %", ascending=False)

display(missing_after)
print("Остаточные пропуски в Rating и DeliveryDate оставлены намеренно; флаги показывают, ожидалось ли значение по статусу заказа.")

### Сохранение очищенной копии

In [ ]:
OUTPUT_PATH = "data/shopease_orders_missing_cleaned.csv"
df_clean.to_csv(OUTPUT_PATH, index=False)
print(f"Очищенная копия сохранена: {OUTPUT_PATH}")